# 🎙️ GPT-SoVITS 파인튜닝 — 육성 전용 모델 학습 (haesollo)

**이 노트북이 맞는지 확인:** 첫 셀 실행 시 `[sovits-v1]` 이 출력됩니다.

## 시작 전 필수 1가지

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 저장. (GPU 없이는 학습 불가)

## 진행 순서 (총 30~40분 예상)

1. **1번 셀 실행** → `dataset` 폴더가 생김 → 왼쪽 폴더 아이콘에서 `dataset` 폴더에
   `haesollo2_full.wav` 와 `ref_finetune.wav` 두 파일을 끌어다 업로드 (PC 위치: `video/output/voice_dataset/`)
2. **2번 셀 실행** → 세션이 한 번 자동 재시작됨 (정상! 놀라지 마세요)
3. 재시작 후 **3번 셀 실행** → 설치 10~15분
4. **4번 셀 실행** → 나오는 `https://xxxx.gradio.live` 링크 클릭 → 웹 화면에서 아래 '클릭 가이드'대로 진행


In [ ]:
# ── 1번 셀: 버전 표식 + 업로드 폴더 준비 ──
print("[sovits-v1] GPT-SoVITS 파인튜닝 노트북 — 2026-07-28")
import os
os.makedirs("/content/dataset", exist_ok=True)
for f in ["haesollo2_full.wav", "ref_finetune.wav"]:
    p = f"/content/dataset/{f}"
    print(f"  {f}: {'업로드됨 ✓ (' + str(os.path.getsize(p)//1024) + ' KB)' if os.path.exists(p) else '아직 없음 — 왼쪽 폴더 아이콘 > dataset 에 끌어다 놓으세요'}")
print("(업로드는 2~3번 셀이 도는 동안 해도 됩니다. 이 셀을 다시 실행하면 업로드 여부를 재확인합니다.)")

In [ ]:
# ── 2번 셀: 콘다 설치 (실행 후 세션이 한 번 자동 재시작됨 — 정상) ──
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")

In [ ]:
# ── 3번 셀: GPT-SoVITS 설치 (10~15분 소요) ──
%%writefile /content/setup.sh
set -e

cd /content

git clone https://github.com/RVC-Boss/GPT-SoVITS.git

cd GPT-SoVITS

if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    :
else
    conda create -n GPTSoVITS "python=3.10.*=*_cpython" -c conda-forge -y
fi

# CPython 고정핀 — 이후 conda 설치가 파이썬을 변종(GraalPy)으로 바꿔치기하는 사고 방지 (2026-07-28 실사고)
echo 'python 3.10.* *_cpython' > /usr/local/envs/GPTSoVITS/conda-meta/pinned

source activate GPTSoVITS

pip install ipykernel

bash install.sh --device CU126 --source HF --download-uvr5

# 버전 상한 미지정 사고 방지 2건 (2026-07-28 실사고):
# ① 최신 fastapi/starlette 1.0이 gradio 화면 렌더를 깨뜨림 → 0.115.2 고정
# ② PyPI 최신 torchaudio가 CUDA 13용이라 cu126 torch와 짝짝이(libcudart.so.13 없음) → cu126 판으로 교체
pip install "fastapi[standard]==0.115.2"
pip install torchaudio --index-url "https://download.pytorch.org/whl/cu126"

In [ ]:
# ── 3-2번 셀: 설치 실행 ──
!cd /content && bash setup.sh

In [ ]:
# ── 3-3번 셀(복구용): 3-2에서 "No matching distribution found for torch" 에러가 났을 때만 실행 ──
# 원인: conda가 환경의 파이썬을 변종(GraalPy)으로 바꿔치기 → 표준 CPython으로 되돌리고 설치 재개
%%bash
set -e
source activate GPTSoVITS
echo 'python 3.10.* *_cpython' > /usr/local/envs/GPTSoVITS/conda-meta/pinned
conda install -y -c conda-forge 'python=3.10.*=*_cpython'
python -c "import platform; print('파이썬 종류:', platform.python_implementation(), platform.python_version())"
cd /content/GPT-SoVITS
bash install.sh --device CU126 --source HF

In [ ]:
# ── 3-4번 셀(복구용): 4번 셀 실행 후 화면이 안 열리고 "unhashable type: 'dict'" 에러가 났을 때만 실행 ──
# 원인: fastapi/starlette 최신판(1.0)이 gradio 화면 렌더 문법을 깨뜨림 → 검증된 버전으로 되돌림
# 실행 전 4번 셀을 정지(■)하고, 이 셀 완료 후 4번 셀을 다시 실행
%%bash
source activate GPTSoVITS
pip install "fastapi[standard]==0.115.2"
python -c "import fastapi, starlette; print('fastapi', fastapi.__version__, '/ starlette', starlette.__version__)"

In [ ]:
# ── 4번 셀: 웹 화면 띄우기 → 출력에 나오는 gradio.live 링크 클릭 ──
!cd /content/GPT-SoVITS && source activate GPTSoVITS && export is_share=True && python webui.py

## 🖱️ 웹 화면 클릭 가이드 (화면 문구는 영어 또는 중국어일 수 있음 — 순서 기준으로 따라오세요)

### A. 데이터셋 만들기 — `0-Fetch dataset` 탭

1. **자르기(0b, Slicing / 语音切分)**: 입력 경로에 `/content/dataset/haesollo2_full.wav` 입력,
   나머지 기본값 → **Start slicing** 버튼. (출력은 자동으로 `output/slicer_opt`)
2. **받아쓰기(0c, ASR / 语音识别)**: ASR 모델 = **faster whisper**, 언어 = **ko**,
   입력 폴더 = `output/slicer_opt` (기본값) → **Start ASR**. (몇 분 걸림. 출력: `output/asr_opt/slicer_opt.list`)
3. 0a(반주 제거)와 0d(라벨 교정)는 건너뜁니다 — 우리 녹음은 목소리만 있는 깨끗한 파일입니다.

### B. 학습 — `1-GPT-SOVITS-TTS` 탭

1. 맨 위 **실험 이름(Experiment name)** 에 `haesollo` 입력. 버전 선택은 기본값 유지.
2. **1A(Dataset formatting)**: 라벨 파일 = `output/asr_opt/slicer_opt.list`,
   오디오 폴더 = `output/slicer_opt` → **One-click formatting** (一键三连) 버튼.
3. **1B(Fine-tuned training)**: 기본값 그대로 → **Start SoVITS training** 완료 후 →
   **Start GPT training**. (각각 몇 분~십몇 분)
4. **1C(Inference)**: **refresh** 버튼으로 모델 목록 새로고침 → GPT 모델 `haesollo-e15` 류,
   SoVITS 모델 `haesollo_e8` 류 선택 → **Open TTS Inference WebUI** 체크 → 새 gradio 링크가 4번 셀 출력에 추가로 뜸 → 클릭.

### C. 음성 생성 (새로 열린 추론 화면)

1. **참조 오디오 업로드**: `/content/dataset/ref_finetune.wav` (왼쪽 폴더에서 내려받아 업로드하거나 경로 입력)
2. **참조 오디오의 텍스트**: `헬로? 아니요. 엘, 오. 딱 두 글자였습니다.` / 참조 언어: **한국어(Korean/韩文)**
3. **생성할 텍스트**(A/B 비교용 — 기존 Chatterbox판과 같은 문장):
   `이 경고, 오십오 년이 지난 지금도 유효합니다. 이천이십이 년에 남태평양의 통가는 해저 케이블 딱 한 가닥이 끊겼는데, 나라 전체가 오 주 동안 인터넷에서 고립됐어요.`
   언어: **한국어** → **합성(Synthesize)** → 재생해 보고 다운로드.
4. 결과 파일은 이름을 `sovits_test.wav` 로 바꿔 클로드에게 전달 → Chatterbox판과 나란히 비교 판정.

### ⚠️ 주의
- 코랩 무료 세션이 끊기면 처음부터 다시 — 학습 완료 후 **모델 파일 백업**: 왼쪽 폴더에서
  `GPT-SoVITS/GPT_weights*` 와 `GPT-SoVITS/SoVITS_weights*` 안의 `haesollo` 파일들을 다운로드해 두세요.
- 화면 문구가 다르면 무리해서 누르지 말고, 화면 캡처를 클로드에게 보여 주세요.